In [1]:
from google.colab import drive
drive.mount("/content/drive")

%cd /content/drive/MyDrive/TUM/Pratikum/code/sliced_rag/SliceGPTModifications
!ls

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/.shortcut-targets-by-id/1VRkguTVtRkllTlSID9ugy4MHwCacsE_K/Pratikum/code/sliced_rag/SliceGPTModifications
build_and_test.sh   LICENSE	    README.md	 SUPPORT.md
CODE_OF_CONDUCT.md  pipelines	    SECURITY.md  tests
experiments	    pyproject.toml  src		 test.sh


In [2]:
!pip install -e .

Obtaining file:///content/drive/.shortcut-targets-by-id/1VRkguTVtRkllTlSID9ugy4MHwCacsE_K/Pratikum/code/sliced_rag/SliceGPTModifications
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.5/510.5 kB 40.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 44.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.7/76.7 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 130.5 MB/s eta 0:00:00
  Building editable for transformercompression (pyproject.toml) ... done
  Created wheel for transformercompression: filename=transformercompression-0.0.1-0.editable-py3-none-any.w

In [2]:
import slicegpt

In [3]:
import os

BASE_RESULTS_DIR = "/content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments"
LOG_DIR = os.path.join(BASE_RESULTS_DIR, "logs_opt_pca")
MODEL_DIR = os.path.join(BASE_RESULTS_DIR, "models_opt_pca")

os.makedirs(LOG_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

In [10]:
import subprocess
from datetime import datetime

def run_slicegpt(dataset, sparsity):
    log_name = f"{dataset}_s{sparsity:.2f}".replace(".", "p") + ".txt" # 0.25 -> 0p25
    log_path = os.path.join(LOG_DIR, log_name)

    # if os.path.exists(log_path):
    #     print(f"[SKIP] Log already exists for {dataset}, sparsity={sparsity}: {log_path}")
    #     return

    save_dir = os.path.join(MODEL_DIR, f"{dataset}_s{sparsity:.2f}".replace(".", "p"))
    os.makedirs(save_dir, exist_ok=True)

    cmd = [
        "python", "/content/drive/MyDrive/TUM/Pratikum/code/sliced_rag/SliceGPTModifications/experiments/run_slicegpt.py",
        "--model", "facebook/opt-125m",
        "--cal-dataset", dataset,
        "--save-dir", save_dir,
        "--sparsity", str(sparsity),
        "--device", "cuda:0",
        "--eval-baseline",
        "--no-wandb",
    ]

    print("\n=====================================================")
    print("Running:", " ".join(cmd))
    print("Log file:", log_path)
    print("Start:", datetime.now())
    print("=====================================================\n")

    with open(log_path, "w") as f:
        # Stream output to both cell and file
        process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in process.stdout:
            print(line, end="")   # to notebook
            f.write(line)         # to log file

    ret = process.wait()
    print("\nFinished with return code:", ret)
    print("End:", datetime.now())


In [12]:
datasets = ["squad", "wikitext2"]
sparsities = [0.0, 0.10, 0.25, 0.4, 0.6]

for dataset in datasets:
    for s in sparsities:
        run_slicegpt(dataset, s)


Running: python /content/drive/MyDrive/TUM/Pratikum/code/sliced_rag/SliceGPTModifications/experiments/run_slicegpt.py --model facebook/opt-125m --cal-dataset squad --save-dir /content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/models_opt_pca/squad_s0p00 --sparsity 0.0 --device cuda:0 --eval-baseline --no-wandb
Log file: /content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/logs_opt_pca/squad_s0p00.txt
Start: 2026-01-12 18:04:49.905644

Running SliceGPT experiment.
PyTorch device: cuda:0
Number of available cuda devices: 1
Loading facebook/opt-125m config and model weights from Hugging Face
Loading model done
Loading dataset: squad
Loading dataset done
Preparing dataloader
Preparing dataloader done
Preparing test dataloader
Preparing test dataloader done
Evaluating perplexity...
Time spent on evaluation: 00:00:43.2180
Original ppl: 28.3798
Replacing layers
Replacing layers done
Fusing layernorm modules
Fusing layernorm modules done
Original model parameters: 163,860,064
New embed